In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

1. Import Library dan Import Dataset

In [5]:
df = pd.read_csv('student_data.csv')

2. Pemeriksaan Duplikasi dan Nilai Hilang (Missing Values)

Missing Value

In [6]:
print("Jumlah Missing Values per Kolom:")
print(df.isnull().sum())

Jumlah Missing Values per Kolom:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64


Duplikasi Data

In [8]:
dup_count = df.duplicated().sum()
unique_count = len(df) - dup_count
df = df.drop_duplicates()

3. Binary Encoding (Variabel Kategorikal Biner)

In [4]:
binary_map = {'yes': 1, 'no': 0}
binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
for col in binary_cols:
    df[col] = df[col].map(binary_map)

df['sex'] = df['sex'].map({'F': 0, 'M': 1})
df['address'] = df['address'].map({'U': 1, 'R': 0})          # Urban: 1, Rural: 0
df['famsize'] = df['famsize'].map({'GT3': 1, 'LE3': 0})       # Greater than 3: 1, Less/Equal 3: 0
df['Pstatus'] = df['Pstatus'].map({'T': 1, 'A': 0})          # Together: 1, Apart: 0
df['school'] = df['school'].map({'GP': 1, 'MS': 0})           # Gabriel Pereira: 1, Mousinho da Silveira: 0

4. Penanganan Outlier pada Kolom Absences (IQR Capping)

In [5]:
Q1 = df['absences'].quantile(0.25)
Q3 = df['absences'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Capping nilai ekstrim absences agar tidak merusak performa model
df['absences'] = np.where(df['absences'] > upper_bound, upper_bound, df['absences'])

5. One-Hot Encoding (Variabel Kategorikal Multi-Kategori)


In [6]:
df_clean = pd.get_dummies(df, columns=['Mjob', 'Fjob', 'reason', 'guardian'], drop_first=True, dtype=int)

6. Feature Engineering (Pembuatan Fitur Baru)

In [7]:
df_clean['parent_edu'] = (df_clean['Medu'] + df_clean['Fedu']) / 2
df_clean['total_alc'] = df_clean['Dalc'] + df_clean['Walc']
df_clean['academic_habit_score'] = df_clean['studytime'] - (df_clean['failures'] * 0.5)
df_clean['passed'] = (df_clean['G3'] >= 10).astype(int)

7. Feature Scaling / Normalisasi Data Numerik

In [8]:
scaler = StandardScaler()
numerical_features = ['age', 'absences', 'G1', 'G2', 'parent_edu', 'total_alc', 'academic_habit_score']
df_clean[numerical_features] = scaler.fit_transform(df_clean[numerical_features])

8. Ekspor Dataset Bersih (Final Output)

In [9]:
df_clean.to_csv('student_data_cleaned.csv', index=False)
print("Pembersihan data selesai. Data disimpan ke 'student_data_cleaned.csv'.")

Pembersihan data selesai. Data disimpan ke 'student_data_cleaned.csv'.
